# **Notebook 3: Baseline Model Evaluation**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] Internet access (to download the model)
- [ ] GPU runtime enabled (Runtime → Change runtime type → T4 GPU)

**Files this notebook will CREATE:**
- [ ] `outputs.json` — `test_query`, `ground_truth`, `baseline_output` _(Required by NB4, NB5, NB7)_

---

In [1]:
# ============================================================
# COLAB PROJECT SETUP
# ============================================================

from google.colab import drive
from pathlib import Path
import os

# Mount Google Drive
drive.mount("/content/drive")

# Permanent project folder in Google Drive
DRIVE_PROJECT_DIR = Path(
    "/content/drive/MyDrive/Hybrid_RAG_Customer_Support"
)

# Temporary workspace for the current Colab runtime
LOCAL_PROJECT_DIR = Path(
    "/content/Hybrid_RAG_Customer_Support"
)

LOCAL_PROJECT_DIR.mkdir(parents=True, exist_ok=True)

# Work from the temporary Colab directory
os.chdir(LOCAL_PROJECT_DIR)

print("Google Drive project:", DRIVE_PROJECT_DIR)
print("Local Colab workspace:", LOCAL_PROJECT_DIR)
print("Current working directory:", Path.cwd())

Mounted at /content/drive
Google Drive project: /content/drive/MyDrive/Hybrid_RAG_Customer_Support
Local Colab workspace: /content/Hybrid_RAG_Customer_Support
Current working directory: /content/Hybrid_RAG_Customer_Support


In [2]:
# ============================================================
# VERIFY GPU RUNTIME
# ============================================================

import torch

print("PyTorch version :", torch.__version__)
print("CUDA available  :", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU runtime is not enabled. In Colab, go to "
        "Runtime → Change runtime type → T4 GPU."
    )

gpu_name = torch.cuda.get_device_name(0)
gpu_memory_gb = (
    torch.cuda.get_device_properties(0).total_memory
    / 1024**3
)

print("GPU detected    :", gpu_name)
print(f"GPU memory      : {gpu_memory_gb:.2f} GB")
print("CUDA version    :", torch.version.cuda)
print("\nGPU runtime is enabled successfully.")

PyTorch version : 2.11.0+cu128
CUDA available  : True
GPU detected    : Tesla T4
GPU memory      : 14.56 GB
CUDA version    : 12.8

GPU runtime is enabled successfully.


## **Stage 3: Solution V1 (Retrieval-Assisted Generation)**

### **Task 3.1: Establish Baseline Performance**

#### **3.1.1 Execute Baseline Inference [2 marks]**
**The Task:** Load the pre-trained base model in 4-bit quantization and generate a response to an ambiguous shipping-delay query without any context.

**Hints & Tips:**
* Use `do_sample=False` for deterministic output. Do NOT pair `temperature=0.0` with `do_sample=False` — it throws a deprecation warning. Use `temperature=None, top_p=None`.
* `BitsAndBytesConfig(load_in_4bit=True)` shrinks the 1.5B model to ~750MB VRAM.
* `max_new_tokens=120` gives room for a complete answer.

**Model Selection:**
* **Qwen/Qwen2.5-1.5B-Instruct** (recommended) — must match what you used in NB2.
* **TinyLlama-1.1B-Chat** — lighter, weaker structured output.
* **Llama-3-8B-Instruct** — best quality, may OOM on free T4 during fine-tuning.

**Learner Inference:** This establishes your zero-shot baseline. Every later improvement is measured against this exact output.

In [3]:
# YOUR CODE HERE
!pip install -q -U transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 99.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.1 MB/s eta 0:00:00


In [4]:
# YOUR CODE HERE
import random
import numpy as np
import torch

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
RANDOM_STATE = 42
MAX_NEW_TOKENS = 120

# Reproducibility
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
torch.cuda.manual_seed_all(RANDOM_STATE)

if not torch.cuda.is_available():
    raise RuntimeError(
        "A GPU runtime is required. Enable the T4 GPU runtime."
    )

In [5]:
# Configure 4-bit quantisation

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

# Load tokenizer and baseline model
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    use_fast=True
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)

model.eval()

print("Baseline model loaded successfully")
print("-" * 50)
print(f"Model ID          : {MODEL_ID}")
print(f"GPU               : {torch.cuda.get_device_name(0)}")
print(f"Model device      : {model.device}")
print(
    f"Model footprint   : "
    f"{model.get_memory_footprint() / 1024**3:.2f} GB"
)
print(f"4-bit quantisation: {model.is_loaded_in_4bit}")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Baseline model loaded successfully
--------------------------------------------------
Model ID          : Qwen/Qwen2.5-1.5B-Instruct
GPU               : Tesla T4
Model device      : cuda:0
Model footprint   : 1.05 GB
4-bit quantisation: True


In [6]:
# Define an ambiguous shipping-delay query
test_query = (
    "My order was supposed to arrive already, but it is still "
    "not here. What is happening and when will I receive it?"
)

messages = [
    {
        "role": "system",
        "content": (
            "You are a customer-support assistant. "
            "Answer the customer's question clearly and concisely."
        )
    },
    {
        "role": "user",
        "content": test_query
    }
]

# Apply the Qwen chat template
prompt_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

model_inputs = tokenizer(
    prompt_text,
    return_tensors="pt",
    add_special_tokens=False
).to(model.device)

In [7]:
# Run deterministic baseline inference
with torch.inference_mode():
    generated_output = model.generate(
        **model_inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        temperature=None,
        top_p=None,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

# Remove the input prompt tokens and decode only the new response
generated_tokens = generated_output[
    0,
    model_inputs["input_ids"].shape[1]:
]

baseline_output = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
).strip()

# Display the zero-shot baseline result
print("\nBaseline inference result")
print("=" * 70)
print(f"Customer query:\n{test_query}")
print("\nBaseline output:")
print(baseline_output)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



Baseline inference result
Customer query:
My order was supposed to arrive already, but it is still not here. What is happening and when will I receive it?

Baseline output:
I'm sorry you're experiencing this delay with your order. Please check your shipping confirmation email or contact our customer service team for more information on the status of your shipment. They should be able to provide an estimated delivery date based on the current transit time. If there has been any issue with tracking or if the item is out of stock at the store where you placed the order, they can also help resolve that. Thank you for understanding.


#### **3.1.2 Evaluate Baseline Quality [2 marks]**
**The Task:** Assess the baseline output for factual inaccuracies against the ground-truth SOP rule.

**Hints & Tips:**
* Compare against the known rule: "Domestic orders deliver within 3-7 business days."
* Did the model invent a timeline? Mention a non-existent tracking system or department?
* Document every hallucination — it justifies Stages 3 and 4.

**Learner Inference:** This hallucination is exactly why you build Stage 3 (a database) and Stage 4 (a router).

In [8]:
# YOUR CODE HERE
import re
import pandas as pd
from IPython.display import display

GROUND_TRUTH_RULE = (
    "Domestic orders are delivered within 3–7 business days."
)

REPEAT_RUNS = 3


def generate_baseline_response():
    """Generate one deterministic baseline response."""

    with torch.inference_mode():
        output_ids = model.generate(
            **model_inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    new_tokens = output_ids[
        0,
        model_inputs["input_ids"].shape[1]:
    ]

    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()


In [9]:
# Repeat deterministic inference
repeated_outputs = [
    generate_baseline_response()
    for _ in range(REPEAT_RUNS)
]

unique_outputs = set(repeated_outputs)

is_consistent = len(unique_outputs) == 1
consistency_rate = (
    repeated_outputs.count(repeated_outputs[0])
    / REPEAT_RUNS
    * 100
)

# Check alignment with the known SOP rule
normalised_output = baseline_output.lower()

supported_timeline_pattern = (
    r"\b3\s*(?:-|–|—|to)\s*7\s+business\s+days?\b"
)

mentions_supported_timeline = bool(
    re.search(
        supported_timeline_pattern,
        normalised_output
    )
)

# Detect numeric delivery timelines appearing in the response
timeline_pattern = (
    r"\b\d+\s*(?:-|–|—|to)\s*\d+\s+"
    r"(?:business\s+)?days?\b"
)

mentioned_timelines = re.findall(
    timeline_pattern,
    normalised_output
)

unsupported_timelines = [
    timeline
    for timeline in mentioned_timelines
    if not re.search(
        supported_timeline_pattern,
        timeline
    )
]

In [13]:
# Flag potentially fabricated systems or procedures
unsupported_claim_patterns = {
    "Invented tracking or communication process": [
        r"shipping confirmation email",
        r"issue with tracking",
        r"tracking portal",
        r"tracking system",
        r"tracking dashboard"
    ],
    "Invented department or support process": [
        r"customer service team",
        r"shipping department",
        r"logistics department",
        r"delivery department",
        r"shipping team"
    ],
    "Unsupported delivery-estimate procedure": [
        r"provide an estimated delivery date",
        r"based on the current transit time"
    ],
    "Invented cause of delay": [
        r"out of stock",
        r"store where you placed the order"
    ],
    "Unsupported carrier procedure": [
        r"contact the carrier",
        r"call the courier",
        r"carrier investigation"
    ],
    "Unsupported refund or compensation": [
        r"automatic refund",
        r"full refund",
        r"compensation",
        r"store credit"
    ]
}

detected_unsupported_claims = []

for claim_type, patterns in unsupported_claim_patterns.items():
    matched_phrases = [
        pattern
        for pattern in patterns
        if re.search(pattern, normalised_output)
    ]

    if matched_phrases:
        detected_unsupported_claims.append(
            {
                "claim_type": claim_type,
                "matched_phrases": ", ".join(matched_phrases)
            }
        )

for timeline in unsupported_timelines:
    detected_unsupported_claims.append(
        {
            "claim_type": "Unsupported delivery timeline",
            "matched_phrases": timeline
        }
    )

# Calculate baseline quality indicators
hallucination_detected = (
    len(detected_unsupported_claims) > 0
)

hallucination_frequency = (
    100.0 if hallucination_detected else 0.0
)

functional_correctness = (
    mentions_supported_timeline
    and not hallucination_detected
)

baseline_evaluation = {
    "ground_truth_rule": GROUND_TRUTH_RULE,
    "mentions_supported_timeline": mentions_supported_timeline,
    "functional_correctness": functional_correctness,
    "deterministic_consistency": is_consistent,
    "consistency_rate": consistency_rate,
    "hallucination_detected": hallucination_detected,
    "hallucination_frequency": hallucination_frequency,
    "unsupported_claim_count": len(
        detected_unsupported_claims
    )
}

In [14]:
# Display the evaluation
print("Baseline quality evaluation")
print("=" * 70)
print(f"Ground-truth SOP rule:\n{GROUND_TRUTH_RULE}")
print(f"\nBaseline output:\n{baseline_output}")

print("\nEvaluation summary")
print("-" * 45)
print(
    f"Supported 3–7-day timeline mentioned : "
    f"{mentions_supported_timeline}"
)
print(
    f"Functionally correct                 : "
    f"{functional_correctness}"
)
print(
    f"Deterministic consistency            : "
    f"{is_consistent}"
)
print(
    f"Consistency rate                     : "
    f"{consistency_rate:.2f}%"
)
print(
    f"Hallucination detected               : "
    f"{hallucination_detected}"
)
print(
    f"Hallucination frequency              : "
    f"{hallucination_frequency:.2f}%"
)

if detected_unsupported_claims:
    print("\nUnsupported claims identified:")
    display(pd.DataFrame(detected_unsupported_claims))
else:
    print("\nNo unsupported claims were detected by the rule-based audit.")

print("\nRepeated deterministic outputs:")

display(
    pd.DataFrame({
        "run": range(1, REPEAT_RUNS + 1),
        "output": repeated_outputs
    })
)

Baseline quality evaluation
Ground-truth SOP rule:
Domestic orders are delivered within 3–7 business days.

Baseline output:
I'm sorry you're experiencing this delay with your order. Please check your shipping confirmation email or contact our customer service team for more information on the status of your shipment. They should be able to provide an estimated delivery date based on the current transit time. If there has been any issue with tracking or if the item is out of stock at the store where you placed the order, they can also help resolve that. Thank you for understanding.

Evaluation summary
---------------------------------------------
Supported 3–7-day timeline mentioned : False
Functionally correct                 : False
Deterministic consistency            : True
Consistency rate                     : 100.00%
Hallucination detected               : True
Hallucination frequency              : 100.00%

Unsupported claims identified:


,claim_type,matched_phrases
0,Invented tracking or communication process,"shipping confirmation email, issue with tracking"
1,Invented department or support process,customer service team
2,Unsupported delivery-estimate procedure,"provide an estimated delivery date, based on t..."
3,Invented cause of delay,"out of stock, store where you placed the order"



Repeated deterministic outputs:


,run,output
0,1,I'm sorry you're experiencing this delay with ...
1,2,I'm sorry you're experiencing this delay with ...
2,3,I'm sorry you're experiencing this delay with ...


### Baseline Evaluation Interpretation

The baseline model produced the same response across all three deterministic runs, resulting in a consistency rate of **100%**. This confirms that the inference configuration is reproducible.

However, the response was **not functionally correct** because it did not mention the ground-truth SOP rule that domestic orders are delivered within **3–7 business days**.

The response also introduced unsupported assumptions, including checking a shipping-confirmation email, contacting a customer-service team, obtaining an estimated date from current transit information, and the item potentially being out of stock at a store. These details were not supported by the supplied ground-truth rule. Therefore, hallucination was detected, giving a hallucination frequency of **100% for this single baseline test case**.

The result demonstrates that deterministic generation can be stable while still being incomplete and unsupported. Retrieval grounding is therefore required in Solution V1 to provide the model with the applicable shipping-delay policy.

---
## Save Artifacts for Downstream Notebooks

**IMPORTANT:** Saves the baseline output. Notebooks 4, 5, and 7 depend on this file.

In [15]:
# YOUR CODE HERE
import json

NOTEBOOK_3_ARTIFACT_DIR = (
    DRIVE_PROJECT_DIR
    / "artifacts"
    / "notebook_3"
)

OUTPUT_PATH = (
    NOTEBOOK_3_ARTIFACT_DIR
    / "outputs.json"
)

NOTEBOOK_3_ARTIFACT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

baseline_artifact = {
    "model_id": MODEL_ID,
    "test_query": test_query,
    "ground_truth": GROUND_TRUTH_RULE,
    "baseline_output": baseline_output,
    "evaluation": {
        "functional_correctness": bool(
            baseline_evaluation["functional_correctness"]
        ),
        "mentions_supported_timeline": bool(
            baseline_evaluation["mentions_supported_timeline"]
        ),
        "deterministic_consistency": bool(
            baseline_evaluation["deterministic_consistency"]
        ),
        "consistency_rate": float(
            baseline_evaluation["consistency_rate"]
        ),
        "hallucination_detected": bool(
            baseline_evaluation["hallucination_detected"]
        ),
        "hallucination_frequency": float(
            baseline_evaluation["hallucination_frequency"]
        ),
        "unsupported_claim_count": int(
            baseline_evaluation["unsupported_claim_count"]
        )
    }
}

with open(OUTPUT_PATH, "w", encoding="utf-8") as file:
    json.dump(
        baseline_artifact,
        file,
        indent=2,
        ensure_ascii=False
    )

# Verify that the saved file can be loaded successfully
with open(OUTPUT_PATH, "r", encoding="utf-8") as file:
    saved_artifact = json.load(file)

required_keys = {
    "test_query",
    "ground_truth",
    "baseline_output"
}

assert required_keys.issubset(saved_artifact.keys())
assert saved_artifact["test_query"] == test_query
assert saved_artifact["ground_truth"] == GROUND_TRUTH_RULE
assert saved_artifact["baseline_output"] == baseline_output

print("Baseline artefact saved successfully")
print("-" * 50)
print(f"Output path:\n{OUTPUT_PATH}")
print(f"\nSaved fields: {list(saved_artifact.keys())}")
print("\nSaved baseline artefact:")
print(
    json.dumps(
        saved_artifact,
        indent=2,
        ensure_ascii=False
    )
)

Baseline artefact saved successfully
--------------------------------------------------
Output path:
/content/drive/MyDrive/Hybrid_RAG_Customer_Support/artifacts/notebook_3/outputs.json

Saved fields: ['model_id', 'test_query', 'ground_truth', 'baseline_output', 'evaluation']

Saved baseline artefact:
{
  "model_id": "Qwen/Qwen2.5-1.5B-Instruct",
  "test_query": "My order was supposed to arrive already, but it is still not here. What is happening and when will I receive it?",
  "ground_truth": "Domestic orders are delivered within 3–7 business days.",
  "baseline_output": "I'm sorry you're experiencing this delay with your order. Please check your shipping confirmation email or contact our customer service team for more information on the status of your shipment. They should be able to provide an estimated delivery date based on the current transit time. If there has been any issue with tracking or if the item is out of stock at the store where you placed the order, they can also help 

---
## END-OF-NOTEBOOK CHECKLIST

> **IMPORTANT: Verify before proceeding to Notebook 4.**

- [x] Base model loaded in 4-bit without errors
- [x] Baseline output generated for `test_query`
- [x] Hallucination assessment documented
- [x] **`outputs.json` saved** with `test_query`, `ground_truth`, `baseline_output` ← _CRITICAL for NB4, 5, 7_
- [x] GPU runtime enabled

**If any item is unchecked, fix it before moving on.**